# Phase 2 — Independent Risk Prediction Evaluation V1
**Purpose:** independently audit saved predictions (no model retraining). 3 targets: schedule, scope, dependency. 14-day outcome labels.

**Interpretation:** validation thresholds were selected on validation; validation F1 is selection-biased. The test projects were previously examined during development, so test results are **exploratory**, not a fresh unbiased holdout. This audit does not establish label construct validity or full temporal leakage safety.

In [1]:
from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (precision_score, recall_score, f1_score, confusion_matrix,
    average_precision_score, roc_auc_score, brier_score_loss, log_loss)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Colab not detected; configure ROOT to local copy.')
ROOT=Path('/content/drive/MyDrive/GC1/Temporal_snapshots')
MODEL=ROOT/'model_training_efficient_investigation'
PRED=MODEL/'predictions'
FEATURE=ROOT/'feature_engineering_output'/'tabpfn_labelled_dataset.parquet'
SPLIT=ROOT/'model_training_output'/'project_split_manifest.csv'
OUT=Path('/content/drive/MyDrive/02_Risk_Prediction/Results/Model_Evaluation_V1')
OUT.mkdir(parents=True,exist_ok=True)
TARGETS=['schedule_risk_label','scope_risk_label','dependency_risk_label']
KEYS=['instance_id','week_number','checkpoint_time']
paths={'validation_predictions':PRED/'validation_predictions.parquet',
       'exploratory_test_predictions':PRED/'exploratory_test_predictions.parquet',
       'labelled_dataset':FEATURE,'project_split_manifest':SPLIT}
missing={k:str(p) for k,p in paths.items() if not p.is_file()}
if missing: raise FileNotFoundError('Missing required upstream files: '+json.dumps(missing,indent=2))
print('All four required files found. Output:',OUT)

Mounted at /content/drive
All four required files found. Output: /content/drive/MyDrive/02_Risk_Prediction/Results/Model_Evaluation_V1


## 1. Load and validate the input contracts

In [2]:
val=pd.read_parquet(paths['validation_predictions'])
test=pd.read_parquet(paths['exploratory_test_predictions'])
data=pd.read_parquet(FEATURE)
split=pd.read_csv(SPLIT)
assert {'instance_id','partition'}.issubset(split.columns)
split['partition']=split['partition'].astype(str).str.lower().replace({'val':'validation'})
assert set(split.partition)=={'train','validation','test'}
assert split.instance_id.is_unique
assert set(split.instance_id)==set(data.instance_id)
assert set(KEYS+TARGETS).issubset(data.columns)
assert not data.duplicated(KEYS).any()
for name,df in [('validation',val),('test',test)]:
    needed=set(KEYS+['risk','actual_label','predicted_label','probability','threshold'])
    assert needed.issubset(df.columns),f'{name}: missing {needed-set(df.columns)}'
    assert not df.duplicated(KEYS+['risk']).any(),f'{name}: duplicate predictions'
    assert set(df.risk)==set(TARGETS)
    assert df.actual_label.isin([0,1]).all() and df.predicted_label.isin([0,1]).all()
    assert np.isfinite(df.probability).all() and df.probability.between(0,1).all()
    assert np.isfinite(df.threshold).all() and df.threshold.between(0,1).all()
    assert (df.predicted_label.to_numpy()==(df.probability.to_numpy()>=df.threshold.to_numpy()).astype(int)).all()
    assert df.groupby(KEYS,dropna=False).risk.nunique().eq(3).all()
    assert df.risk.value_counts().eq(len(df)//3).all()
    assert df.instance_id.isin(split.loc[split.partition==name,'instance_id']).all()
    print(name, 'prediction rows:',len(df),'checkpoints:',len(df)//3,'instances:',df.instance_id.nunique())
assert set(val.instance_id).isdisjoint(set(test.instance_id))
assert set(split.loc[split.partition=='train','instance_id']).isdisjoint(set(val.instance_id)|set(test.instance_id))
assert len(val)==198*3 and len(test)==197*3
assert len(data)==987 and data.instance_id.nunique()==30
print('PASS: basic schema, 18/6/6 instance split, uniqueness and threshold consistency')

validation prediction rows: 594 checkpoints: 198 instances: 6
test prediction rows: 591 checkpoints: 197 instances: 6
PASS: basic schema, 18/6/6 instance split, uniqueness and threshold consistency


## 2. Independent source-label and checkpoint alignment

In [8]:
def norm_keys(df):
    x=df.copy();x['instance_id']=x.instance_id.astype(str)
    x['week_number']=pd.to_numeric(x.week_number,errors='raise').astype(int)
    x['checkpoint_time']=pd.to_datetime(x.checkpoint_time,utc=True,format='mixed',errors='raise')
    return x
D=norm_keys(data);S=split.copy();S['instance_id']=S.instance_id.astype(str)
D=D.merge(S[['instance_id','partition']],on='instance_id',validate='many_to_one')
comparison=[]
for name,frame in [('validation',val),('test',test)]:
    p=norm_keys(frame)
    truth=D[KEYS+TARGETS+['partition']].melt(id_vars=KEYS+['partition'],var_name='risk',value_name='source_label')
    merged = p.merge(
      truth,
      on=KEYS + ['risk'],
      validate='one_to_one',
      indicator=True,
      suffixes=('_pred', '_truth')
    )

    assert len(merged) == len(p), name + ' row count mismatch'

    assert merged['_merge'].eq('both').all(), (
        name + ' keys missing'
    )

    assert merged['partition_pred'].eq(name).all(), (
        name + ' prediction split mismatch'
    )

    assert merged['partition_truth'].eq(name).all(), (
        name + ' ground-truth split mismatch'
    )

    assert merged['actual_label'].eq(
        merged['source_label']
    ).all(), name + ' labels mismatch'
    assert merged.actual_label.eq(merged.source_label).all(),name+' labels mismatch'
    comparison.append({'partition':name,'rows':len(p),'label_mismatches':int((merged.actual_label!=merged.source_label).sum())})
display(pd.DataFrame(comparison))
print('PASS: prediction labels agree with labelled dataset at exact checkpoint keys')

,partition,rows,label_mismatches
0,validation,594,0
1,test,591,0


PASS: prediction labels agree with labelled dataset at exact checkpoint keys


In [9]:
print(merged.columns.tolist())

['instance_id', 'week_number', 'checkpoint_time', 'partition_pred', 'risk', 'algorithm', 'actual_label', 'probability', 'threshold', 'predicted_label', 'partition_truth', 'source_label', '_merge']


## 3. Metrics recalculated from raw predictions (metrics are independently recomputed from saved predictions)

In [10]:
def metric_row(g):
    y=g.actual_label.to_numpy(dtype=int);yp=g.predicted_label.to_numpy(dtype=int);p=g.probability.to_numpy(dtype=float)
    tn,fp,fn,tp=confusion_matrix(y,yp,labels=[0,1]).ravel()
    return dict(n=len(y),positives=int(y.sum()),prevalence=float(y.mean()),predicted_positives=int(yp.sum()),
      TN=int(tn),FP=int(fp),FN=int(fn),TP=int(tp),precision=float(precision_score(y,yp,zero_division=0)),
      recall=float(recall_score(y,yp,zero_division=0)),F1=float(f1_score(y,yp,zero_division=0)),
      PR_AUC=float(average_precision_score(y,p)) if y.sum()>0 else np.nan,
      AUROC=float(roc_auc_score(y,p)) if len(np.unique(y))==2 else np.nan,
      Brier=float(brier_score_loss(y,p)),log_loss=float(log_loss(y,p,labels=[0,1])))
rows=[];summary=[]
for name,frame in [('validation',val),('exploratory_test',test)]:
    for risk,g in frame.groupby('risk'):
        rows.append({'partition':name,'risk':risk,**metric_row(g)})
    subset=[r for r in rows if r['partition']==name]
    summary.append({'partition':name,'checkpoints':len(frame)//3,'instances':frame.instance_id.nunique(),
        'macro_F1':float(np.mean([r['F1'] for r in subset])),
        'macro_PR_AUC':float(np.mean([r['PR_AUC'] for r in subset])),
        'macro_AUROC':float(np.mean([r['AUROC'] for r in subset])),
        'macro_Brier':float(np.mean([r['Brier'] for r in subset])),
        'pooled_micro_F1':float(f1_score(frame.actual_label,frame.predicted_label,zero_division=0))})
metrics=pd.DataFrame(rows);overall=pd.DataFrame(summary)
display(metrics.round(4));display(overall.round(4))
metrics.to_csv(OUT/'per_risk_metrics.csv',index=False)
overall.to_csv(OUT/'overall_metrics.csv',index=False)
print('Note: macro metrics average the three separate binary risks; this is multi-label prediction.')

,partition,risk,n,positives,prevalence,predicted_positives,TN,FP,FN,TP,precision,recall,F1,PR_AUC,AUROC,Brier,log_loss
0,validation,dependency_risk_label,198,16,0.0808,84,108,74,6,10,0.1190,0.6250,0.2000,0.1668,0.6223,0.1094,0.3563
1,validation,schedule_risk_label,198,52,0.2626,107,76,70,15,37,0.3458,0.7115,0.4654,0.3142,0.5892,0.1940,0.5743
2,validation,scope_risk_label,198,33,0.1667,65,122,43,11,22,0.3385,0.6667,0.4490,0.3806,0.7486,0.1232,0.3959
3,exploratory_test,dependency_risk_label,197,14,0.0711,59,133,50,5,9,0.1525,0.6429,0.2466,0.1305,0.7521,0.1075,0.3177
4,exploratory_test,schedule_risk_label,197,52,0.2640,95,96,49,6,46,0.4842,0.8846,0.6259,0.5176,0.8095,0.1533,0.4644
5,exploratory_test,scope_risk_label,197,37,0.1878,114,80,80,3,34,0.2982,0.9189,0.4503,0.2972,0.7199,0.1526,0.4534


,partition,checkpoints,instances,macro_F1,macro_PR_AUC,macro_AUROC,macro_Brier,pooled_micro_F1
0,validation,198,6,0.3715,0.2872,0.6533,0.1422,0.3866
1,exploratory_test,197,6,0.4409,0.3151,0.7605,0.1378,0.4798


Note: macro metrics average the three separate binary risks; this is multi-label prediction.


## 4. Thresholds and metric reconciliation with teammate reports

In [11]:
thresholds=pd.concat([val.assign(partition='validation'),test.assign(partition='test')]).groupby(['risk','partition']).threshold.agg(['min','max','nunique']).reset_index()
assert thresholds['nunique'].eq(1).all()
assert thresholds.groupby('risk')['min'].nunique().eq(1).all()
display(thresholds)
thresholds.to_csv(OUT/'threshold_audit.csv',index=False)
for filename,partition in [('validation_metrics_thresholds.csv','validation'),('exploratory_test_metrics.csv','exploratory_test')]:
    path=MODEL/'metrics'/filename
    if not path.exists():
        print('UNAVAILABLE teammate report:',path.name);continue
    orig=pd.read_csv(path)
    joined=metrics[metrics.partition==partition].merge(orig,on='risk',suffixes=('_audit','_saved'))
    issues=[]
    for key in ['F1','precision','recall']:
        if f'{key}_saved' in joined:
            diffs=(joined[f'{key}_audit']-joined[f'{key}_saved']).abs()
            if diffs.gt(1e-6).any():issues.append(key)
    print(filename,'metric disagreements:',issues if issues else 'none')
    if issues: warnings.warn('Discrepancies found; inspect metrics before reporting')

,risk,partition,min,max,nunique
0,dependency_risk_label,test,0.126258,0.126258,1
1,dependency_risk_label,validation,0.126258,0.126258,1
2,schedule_risk_label,test,0.282411,0.282411,1
3,schedule_risk_label,validation,0.282411,0.282411,1
4,scope_risk_label,test,0.147636,0.147636,1
5,scope_risk_label,validation,0.147636,0.147636,1


validation_metrics_thresholds.csv metric disagreements: none
exploratory_test_metrics.csv metric disagreements: none


## 5. Class balance, per-instance variability and calibration bins

In [12]:
project_rows=[];cal_rows=[]
for name,frame in [('validation',val),('exploratory_test',test)]:
    for (risk,instance),g in frame.groupby(['risk','instance_id']):
        project_rows.append({'partition':name,'risk':risk,'instance_id':instance,**metric_row(g)})
    for risk,g in frame.groupby('risk'):
        temp=g.copy();temp['bin']=pd.cut(temp.probability,np.linspace(0,1,11),include_lowest=True)
        for interval,h in temp.groupby('bin',observed=True):
            cal_rows.append({'partition':name,'risk':risk,'probability_bin':str(interval),
                'n':len(h),'mean_probability':float(h.probability.mean()),'observed_rate':float(h.actual_label.mean())})
project_metrics=pd.DataFrame(project_rows);calibration=pd.DataFrame(cal_rows)
display(project_metrics[['partition','risk','instance_id','n','positives','F1','PR_AUC']].round(3))
project_metrics.to_csv(OUT/'per_instance_metrics.csv',index=False)
calibration.to_csv(OUT/'calibration_bins.csv',index=False)
print('Caution: undefined PR-AUC/AUROC on one-class instances is expected; per-instance F1=0 can be misleading.')

,partition,risk,instance_id,n,positives,F1,PR_AUC
0,validation,dependency_risk_label,AEROGEAR-I01,33,5,0.263,0.308
1,validation,dependency_risk_label,JBDS-I01,33,4,0.286,0.156
2,validation,dependency_risk_label,JSDCLOUD-I01,33,0,0.000,NaN
3,validation,dependency_risk_label,MC-I01,33,2,0.000,0.049
4,validation,dependency_risk_label,SAK-I01,33,0,0.000,NaN
5,validation,dependency_risk_label,SPARK-I01,33,5,0.286,0.170
6,validation,schedule_risk_label,AEROGEAR-I01,33,14,0.632,0.494
7,validation,schedule_risk_label,JBDS-I01,33,12,0.286,0.282
8,validation,schedule_risk_label,JSDCLOUD-I01,33,0,0.000,NaN
9,validation,schedule_risk_label,MC-I01,33,22,0.766,0.694


Caution: undefined PR-AUC/AUROC on one-class instances is expected; per-instance F1=0 can be misleading.


## 6. Basic feature leakage screen — not a complete causal/temporal audit

In [13]:
import re
feature_names=pd.read_csv(ROOT/'feature_engineering_output'/'selected_feature_list.csv').feature_name.astype(str).tolist()
patterns=r'(future|outcome|target|label|next_?14|lead_?time|post_?checkpoint|ground_?truth)'
suspicious=[f for f in feature_names if re.search(patterns,f,re.I)]
assert len(feature_names)==34
print('Feature count:',len(feature_names),'Suspicious names:',suspicious)
print('Split isolation PASS at instance_id level; this does NOT establish historical feature correctness.')
print('Need separate source-level audit to ensure each feature uses data available by checkpoint.')
(pd.DataFrame({'feature':feature_names,'name_flag':[bool(re.search(patterns,f,re.I)) for f in feature_names]})
 .to_csv(OUT/'feature_name_leakage_screen.csv',index=False))

Feature count: 34 Suspicious names: []
Split isolation PASS at instance_id level; this does NOT establish historical feature correctness.
Need separate source-level audit to ensure each feature uses data available by checkpoint.


## 7. Interpretation and export

In [14]:
result={'status':'COMPLETED_INDEPENDENT_SAVED_PREDICTION_AUDIT',
 'scope':{'checkpoints':987,'project_instances':30,'validation_instances':6,'exploratory_test_instances':6},
 'primary_metrics':overall.replace({np.nan:None}).to_dict(orient='records'),
 'risk_metrics':metrics.replace({np.nan:None}).to_dict(orient='records'),
 'methodological_limits':['Labels follow operational 14-day rules; construct validity not independently proven.',
 'Validation thresholds were tuned on validation; validation F1 is optimistic.',
 'Exploratory test projects were previously inspected; not a fresh untouched holdout.',
 'Feature name screen and grouped splits do not prove absence of future information.',
 'PR-AUC is average precision; interpret against risk prevalence.',
 'No retraining or new prediction generation was performed.']}
(OUT/'phase2_model_evaluation_summary.json').write_text(json.dumps(result,indent=2,allow_nan=False),encoding='utf-8')
print('Saved:',OUT)
print('Files:',[p.name for p in sorted(OUT.iterdir())])
display(overall.round(4))

Saved: /content/drive/MyDrive/02_Risk_Prediction/Results/Model_Evaluation_V1
Files: ['calibration_bins.csv', 'feature_name_leakage_screen.csv', 'overall_metrics.csv', 'per_instance_metrics.csv', 'per_risk_metrics.csv', 'phase2_model_evaluation_summary.json', 'threshold_audit.csv']


,partition,checkpoints,instances,macro_F1,macro_PR_AUC,macro_AUROC,macro_Brier,pooled_micro_F1
0,validation,198,6,0.3715,0.2872,0.6533,0.1422,0.3866
1,exploratory_test,197,6,0.4409,0.3151,0.7605,0.1378,0.4798
